# LangChain Fundamentals with Groq

**Week 2** · From raw API calls to a tool-using assistant

## What you already know (Week 1)

- The raw `groq` Python SDK
- Message dicts: `{"role": "user", "content": "..."}`
- A basic chatbot REPL

## What you'll learn today

We build **one project**: a general-purpose assistant that gets smarter section by section. Every new LangChain tool shows up only **after the previous approach breaks**. You feel the pain, then you fix it.

```
raw groq pain
  → prompt templates      (hardcoded strings break)
  → LCEL pipe + parser     (manual .content is tedious)
  → sequential chains      (one-shot answers are shallow)
  → custom @tool           (static model knowledge)
  → search tool            (model can't reach the internet)
  → full tool agent        (everything wired together)
  → assignment + 2 challenges
```

> Run cells **top to bottom**. Sections 1–8 are in class. Section 9 is homework.

**A note on the name:** *Groq* (LPU inference provider, `console.groq.com`) is not the same as *Grok* (xAI). This notebook uses **Groq** — same provider as Week 1, free tier, no credit card.

## Section 1 — Setup

Install LangChain + the Groq integration + a free search tool. One install cell for the whole notebook.

> **If you ran an unpinned install first:** after running the cell below, do **Runtime → Restart session**, then run top-to-bottom. Colab keeps the old version loaded in memory until you restart.

In [ ]:
# Pinned to the 0.3 stack on purpose. LangChain v1 removed create_tool_calling_agent
# and AgentExecutor (Section 8) — pinning keeps every cell in this notebook working.
!pip install -q \
  "langchain>=0.3,<1.0" \
  "langchain-core>=0.3,<1.0" \
  "langchain-community>=0.3,<1.0" \
  "langchain-groq>=0.2,<0.3" \
  duckduckgo-search

### API key

Never commit API keys to Git. In Colab, store `GROQ_API_KEY` in the **key icon** sidebar (Secrets), or paste it when prompted.

Get a free key at [console.groq.com](https://console.groq.com).

In [ ]:
import os
import getpass

# Try Colab Secrets first; fall back to a prompt.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

### One shared LLM for the whole notebook

`ChatGroq` is LangChain's wrapper around the Groq API. We create it **once** and reuse it everywhere. `temperature=0` keeps answers stable and repeatable in class.

In [ ]:
from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0)

# Quick smoke test
print(llm.invoke("Say hello in exactly five words.").content)

## Section 2 — The Hardcoded Prompt Problem

In Week 1 you wrote the prompt string by hand every time. Watch the pain: to ask about a new topic you **copy-paste and edit the string**. One typo in one copy and your output silently breaks.

In [ ]:
# Naive: hardcoded, not reusable. Imagine doing this 20 times.
print(llm.invoke("Explain recursion in simple terms.").content)
print("---")
print(llm.invoke("Explain recursoin in simple terms.").content)  # typo — nothing warns you

### Fix: `ChatPromptTemplate`

Write the prompt **once** with a `{variable}` slot. Fill the slot at call time. The structure is fixed; only the variable changes.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a clear, concise teacher. Use simple language."),
    ("human", "Explain {topic} in simple terms."),
])

# Inspect what actually gets sent — no guessing.
for msg in prompt.format_messages(topic="recursion"):
    print(f"[{msg.type}] {msg.content}")

In [ ]:
# Pipe the prompt into the llm. Same structure, any topic, zero copy-paste.
chain = prompt | llm
print(chain.invoke({"topic": "recursion"}).content)

### 🔹 Mini-exercise (2 min)

Run the **same template** with two different topics. Fill the TODOs.

In [ ]:
# TODO: pick any two topics you find interesting
topic_a = "binary search"   # TODO: change me
topic_b = "recursion"        # TODO: change me

print("A:", chain.invoke({"topic": topic_a}).content)
print("\nB:", chain.invoke({"topic": topic_b}).content)

## Section 3 — Four Prompt Techniques

Same template machinery, different **instructions**. Each technique nudges the model toward better answers.

| Technique | Key idea |
|-----------|----------|
| Zero-shot | instruction only |
| Few-shot | show one example via an `("ai", ...)` turn |
| Chain-of-thought | ask for step-by-step reasoning |
| Zero-shot CoT | the trigger phrase "Let's think step by step" |

In [ ]:
# 1) Zero-shot — just an instruction
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in one sentence."),
    ("human", "{topic}"),
])
print((zero_shot | llm).invoke({"topic": "What is an API?"}).content)

In [ ]:
# 2) Few-shot — the ("ai", ...) turn shows the model the FORMAT we want
few_shot = ChatPromptTemplate.from_messages([
    ("system", "Define the term in the exact style shown."),
    ("human", "variable"),
    ("ai", "📦 variable — a named box that stores a value."),
    ("human", "{topic}"),
])
print((few_shot | llm).invoke({"topic": "function"}).content)

In [ ]:
# 3) Chain-of-thought — ask for explicit reasoning (good for comparisons/decisions)
cot = ChatPromptTemplate.from_messages([
    ("system", "Show your reasoning step by step, then give a final recommendation."),
    ("human", "Should a beginner learn Python or JavaScript first?"),
])
print((cot | llm).invoke({}).content)

In [ ]:
# 4) Zero-shot CoT — one trigger phrase unlocks reasoning on a tricky question
zero_cot = ChatPromptTemplate.from_messages([
    ("human", "Let's think step by step.\n{question}"),
])
print((zero_cot | llm).invoke({
    "question": "A bat and ball cost $1.10. The bat costs $1 more than the ball. How much is the ball?"
}).content)

### 🔹 Mini-exercise (3 min)

Pick **one** technique above and write your own prompt for a topic you care about.

In [ ]:
# TODO: build your own ChatPromptTemplate using one technique, then invoke it
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),  # TODO: customize
    ("human", "{topic}"),                          # TODO: customize
])
print((my_prompt | llm).invoke({"topic": "explain HTTP in one line"}).content)

## Section 4 — LCEL: The Pipe Operator

Notice you keep typing `.content` to dig the string out of the reply. That's an `AIMessage` object — annoying when you want to pass text to the next step.

In [ ]:
result = (prompt | llm).invoke({"topic": "recursion"})
print(type(result))      # <class '...AIMessage'>
print(result.content)    # the string lives inside .content

### Fix: `StrOutputParser`

Add one more pipe stage. Now the chain returns a **plain string** — ready to compose.

> Anything you can pipe with `|` is a **Runnable**. Prompts, models, and parsers are all Runnables. That's the whole idea behind LCEL (LangChain Expression Language).

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "recursion"})
print(type(result))   # <class 'str'> — no more .content
print(result)

### Multi-turn history with `MessagesPlaceholder`

To remember a conversation, slot prior messages into the prompt with a placeholder.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with a good memory."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

history = [
    ("human", "My favorite color is blue."),
    ("ai", "Got it — blue it is!"),
]
chat_chain = chat_prompt | llm | StrOutputParser()
print(chat_chain.invoke({"history": history, "question": "What is my favorite color?"}))

### 🔹 Mini-exercise (2 min)

Add `StrOutputParser` to your Section 3 chain so it returns a plain string.

In [ ]:
# TODO: rebuild my_prompt's chain with StrOutputParser and confirm the type is str
my_str_chain = my_prompt | llm | StrOutputParser()  # TODO: reuse YOUR prompt from Section 3
out = my_str_chain.invoke({"topic": "explain DNS in one line"})
print(type(out), "→", out)

## Section 5 — Sequential Chains (Two-Step Workflow)

Some jobs need **two steps**. "Summarize this, *then* quiz me on it" in one giant prompt produces a shallow blob. Better: run two chains, feeding the first's output into the second.

No magic here — it's plain Python calling two chains. The summary is a `str`, so it drops straight into the next chain's variable.

In [ ]:
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the topic in 3 short bullet points."),
    ("human", "{topic}"),
])
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write 3 short quiz questions based ONLY on this summary."),
    ("human", "{summary}"),
])

step1 = summarize_prompt | llm | StrOutputParser()
step2 = quiz_prompt | llm | StrOutputParser()

summary = step1.invoke({"topic": "binary search"})
print("=== SUMMARY ===")
print(summary)

questions = step2.invoke({"summary": summary})  # str flows straight in
print("\n=== QUIZ ===")
print(questions)

### 🔹 Mini-exercise (5 min, pairs)

Write a 2-step chain: first **explain a concept**, then produce **3 interview questions** about it.

In [ ]:
# TODO: fill the two prompts, then run both steps
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain the concept clearly in one paragraph."),  # TODO
    ("human", "{concept}"),
])
interview_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write 3 interview questions based on this explanation."),  # TODO
    ("human", "{explanation}"),
])

explain = explain_prompt | llm | StrOutputParser()
ask_qs = interview_prompt | llm | StrOutputParser()

explanation = explain.invoke({"concept": "REST APIs"})  # TODO: your concept
print(explanation)
print("\n---\n")
print(ask_qs.invoke({"explanation": explanation}))

## Section 6 — Custom Tools

Ask the assistant "what's today's date?" — it **guesses or refuses**. The model has no clock, no calculator, no live data. Tools give it real abilities.

In [ ]:
# Pain: the model cannot actually know the date.
print(llm.invoke("What is today's date? Answer with just the date.").content)

### Fix: the `@tool` decorator

A tool is a normal Python function. The **docstring is critical** — the model reads it to decide *when* to call the tool. Write docstrings like instructions to the model.

In [ ]:
from langchain.tools import tool
from datetime import date

@tool
def get_today(query: str = "") -> str:
    """Returns today's date. Use this whenever asked about the current date or day."""
    return f"Today is {date.today().isoformat()}"

@tool
def word_counter(text: str) -> str:
    """Counts the number of words in the given text. Use when asked how many words something has."""
    return f"{len(text.split())} words"

# Test tools directly — they're just functions you call with .invoke()
print(get_today.invoke(""))
print(word_counter.invoke("one two three four five"))

### 🔹 Mini-exercise (5 min)

Write **one** `@tool` of your own. Ideas: `celsius_to_fahrenheit`, `reverse_string`, `list_to_bullets`. Give it a clear docstring — that's how the agent will know to use it later.

In [ ]:
# TODO: write your own tool. Keep the @tool decorator and a clear docstring.
@tool
def my_tool(text: str) -> str:
    """TODO: describe exactly when the model should call this tool."""
    return text[::-1]  # TODO: replace with your logic (this one reverses text)

print(my_tool.invoke("hello"))

## Section 7 — A Search Tool (reach the internet)

The model's knowledge has a **cutoff date**. Ask about something recent and it can't help. `DuckDuckGoSearchRun` is a ready-made tool — no API key needed. It returns a plain string.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
print(search.run("latest stable Python release version"))

> A tool just returns a string. By itself it does nothing smart. The **agent** (next section) is what decides *when* to call which tool.

## Section 8 — The Full Tool Agent

Now wire it together. `create_tool_calling_agent` + `AgentExecutor` run a **ReAct loop**: the model Thinks, picks a Tool, reads the Observation, and repeats until it can Answer.

- `verbose=True` so you can watch the loop.
- `max_iterations` caps the loop so it can't spin forever.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when they help answer the question."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),  # the agent's working memory — required
])

tools = [search, get_today, word_counter]
agent = create_tool_calling_agent(llm, tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

In [ ]:
# Query 1 — triggers the date tool
print(executor.invoke({"input": "What is today's date?"})["output"])

In [ ]:
# Query 2 — triggers the search tool
print(executor.invoke({"input": "What is the latest stable version of Python?"})["output"])

In [ ]:
# Query 3 — combined: needs two tools in one run
print(executor.invoke({
    "input": "How many words are in the phrase 'the quick brown fox', and what is today's date?"
})["output"])

### 🔹 Mini-exercise (5 min)

Add **your** tool from Section 6 to the executor, then write a query that forces the agent to call it.

In [ ]:
# TODO: add my_tool to the tools list and rebuild the executor
my_tools = [search, get_today, word_counter, my_tool]
my_agent = create_tool_calling_agent(llm, my_tools, agent_prompt)
my_executor = AgentExecutor(agent=my_agent, tools=my_tools, verbose=True, max_iterations=5)

# TODO: write a query that clearly needs YOUR tool
print(my_executor.invoke({"input": "Reverse the word 'banana' for me."})["output"])

---
# Section 9 — Assignments

Three tiers. Do them in order.

1. **Warm-ups** — quick reps to lock in each concept. *(in/after class)*
2. **Build assignment** — assemble a full assistant from scratch. *(homework)*
3. **Two challenges** — push past the lesson. *(stretch goals)*

Everything here uses **only** what's above: prompt templates, LCEL, `StrOutputParser`, sequential chains, `@tool`, search, and `AgentExecutor`. No new libraries needed.

## 9A — Warm-ups (get started)

Short and concrete. Each one is a single cell. If a cell runs, you got it.

**W1.** Make a `ChatPromptTemplate` with **two** variables (e.g. `{level}` and `{topic}`) and invoke it.

**W2.** Build a one-line LCEL chain `prompt | llm | StrOutputParser()` and print `type()` of the result to prove it's a `str`.

**W3.** Write a `@tool` named `add_numbers(a_and_b: str)` that parses `"3, 4"` and returns the sum. Test it with `.invoke("3, 4")`.

**W4.** Build a 2-step sequential chain: **define a word** → **use it in a sentence**.

In [ ]:
# W1 — two-variable template
# TODO


In [ ]:
# W2 — LCEL chain returning a str
# TODO


In [ ]:
# W3 — add_numbers tool
# TODO


In [ ]:
# W4 — define → use-in-sentence sequential chain
# TODO


## 9B — Build Assignment: Your Own Assistant

**No starter code.** Build it from scratch using the warm-up pieces.

### Requirements
1. A `ChatPromptTemplate` with at least **one** variable.
2. An LCEL chain ending in `StrOutputParser`.
3. At least **2** custom `@tool` functions (with clear docstrings).
4. An `AgentExecutor` wired with **search + your custom tools**.
5. **3+** test queries that each clearly trigger a different tool.

### Deliverables
- Working notebook cells.
- A **screenshot of `verbose=True` output** showing the tool calls (Thought → Tool → Observation → Answer).
- *Optional:* include a 2-step chain (summarize → quiz, or explain → rewrite).

In [ ]:
# 9B — build your assistant here (add as many cells as you need)
# TODO


## 9C — Challenge 1: The Self-Briefing Research Agent  🧩

**Goal:** combine a **sequential chain** with a **tool agent** so the model researches a topic, then turns its findings into something useful — in one pipeline.

**Build:**
1. A custom `@tool` `headline_search(topic: str)` that calls `DuckDuckGoSearchRun` internally and returns the raw results.
2. An `AgentExecutor` (with `headline_search` + `get_today`) that, given a topic, produces a short **"what's new"** briefing. The agent must call **both** tools — the date *and* the search — in a single run.
3. A **second** LCEL chain that takes the agent's briefing string and rewrites it as a **3-bullet summary for a busy manager**.

**Success looks like:** one function call in → dated briefing → 3 clean bullets out. Print the `verbose=True` trace showing both tools fired.

**Hint:** the agent returns a string under `["output"]`. Feed that string straight into your summarizer chain's variable — exactly like Section 5.

In [ ]:
# 9C — Challenge 1
# TODO: headline_search tool → AgentExecutor (search + date) → manager-summary chain


## 9D — Challenge 2: The Tool That Thinks  🧠

Here's the twist: a tool can **call the LLM itself**. You'll build a tool whose body runs its own LCEL chain, then hand that tool to an agent. The agent won't know — to it, it's just another tool.

**Build:**
1. A `@tool` `explain_like_im_five(concept: str)` whose **body** runs a small `prompt | llm | StrOutputParser()` chain that rewrites any concept for a 5-year-old. (A tool calling the model is allowed — it returns a string like any other tool.)
2. Add it to an `AgentExecutor` alongside `search`.
3. Ask a question that makes the agent **search first, then ELI5 the result** — e.g. *"Find what quantum computing is, then explain it like I'm five."* The agent should chain `search` → `explain_like_im_five` on its own.

**Why it's hard:** you're nesting an LLM chain inside a tool inside an agent. Watch the `verbose` trace — you'll see the agent call your tool, and your tool quietly makes its own model call.

**Gotcha:** keep the tool's docstring sharp ("Use this to re-explain a concept in very simple words") or the agent won't know when to reach for it. Avoid infinite loops — keep `max_iterations` small.

In [ ]:
# 9D — Challenge 2
# TODO: explain_like_im_five tool (runs its own chain) → AgentExecutor (search + tool) → chained query


## Homework checklist

- [ ] Sections 1–8 run top-to-bottom in class
- [ ] **9A** — all four warm-ups run
- [ ] **9B** — assistant built; screenshot of `verbose=True` tool calls saved
- [ ] **9C** — Challenge 1: research agent → manager summary working
- [ ] **9D** — Challenge 2: LLM-inside-a-tool agent working
- [ ] Bring one bug or surprise you hit to next class

**You now know:** prompt templates, prompt techniques, LCEL, output parsers, sequential chains, custom tools, search, and tool agents — the core of LangChain. Next: memory, retrieval, and LangGraph.